# 2일차 — Value-based & Policy-based Methods

2026-07-28 (화) · 신경망 함수 근사를 도입한 DQN 계열과, 정책을 직접 학습하는 Policy Gradient·A2C를 PyTorch로 구현합니다.

> 위에서부터 순서대로 실행하세요. 뒤 교시가 앞 교시의 변수·클래스를 그대로 이어 씁니다.


## 1교시 · DQN 소개

`09:30 ~ 10:30` · `replay_buffer.py`

- 테이블 방식의 한계와 함수 근사의 필요성을 이해한다
- DQN의 핵심 요소인 경험 재현(Replay Buffer)과 타깃 네트워크를 설명할 수 있다


In [ ]:
import random
from collections import deque
import numpy as np
import torch

class ReplayBuffer:
    """경험 재현 버퍼 — DQN, DDPG, SAC 3일 내내 재사용합니다"""
    def __init__(self, capacity=100_000, action_dtype=torch.int64):
        # action_dtype: 오늘 DQN은 행동이 "몇 번 행동"인 정수라 int64입니다.
        # 3일차 DDPG·SAC는 행동이 연속값(실수 벡터)이므로 float32로 바꿔 씁니다
        #   buffer = ReplayBuffer(100_000, action_dtype=torch.float32)
        # int64로 두면 실수 행동이 정수로 잘려 학습이 통째로 망가집니다.
        self.buffer = deque(maxlen=capacity)
        self.action_dtype = action_dtype

    def push(self, s, a, r, s_next, done):
        self.buffer.append((s, a, r, s_next, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, s_next, done = zip(*batch)
        # np.array로 한 번 묶고 텐서로 바꿉니다.
        # 배열 리스트를 텐서로 바로 만들면 파이토치가 하나씩 복사해 매우 느립니다.
        return (torch.as_tensor(np.array(s), dtype=torch.float32),
                torch.as_tensor(np.array(a), dtype=self.action_dtype),
                torch.as_tensor(np.array(r), dtype=torch.float32),
                torch.as_tensor(np.array(s_next), dtype=torch.float32),
                torch.as_tensor(np.array(done), dtype=torch.float32))

    def __len__(self):
        return len(self.buffer)

## 2교시 · Double DQN 소개

`10:30 ~ 11:30` · `ddqn_target.py`

- Q-Learning의 최대화 편향(maximization bias)이 왜 생기는지 이해한다
- Double DQN이 선택과 평가를 분리하는 방식을 설명할 수 있다


In [ ]:
import torch

# DQN vs Double DQN — 목표(target) 계산의 차이
@torch.no_grad()
def dqn_target(q_target, r, s_next, done, gamma=0.99):
    max_q = q_target(s_next).max(dim=1).values          # 타깃넷이 선택+평가
    return r + gamma * max_q * (1 - done)

@torch.no_grad()
def double_dqn_target(q_online, q_target, r, s_next, done, gamma=0.99):
    best_a = q_online(s_next).argmax(dim=1, keepdim=True)   # 선택: 온라인넷
    max_q = q_target(s_next).gather(1, best_a).squeeze(1)   # 평가: 타깃넷
    return r + gamma * max_q * (1 - done)

## 3교시 · PyTorch 소개 및 구현

`11:30 ~ 12:30` · `pytorch_basics.py`

- Tensor, Autograd, nn.Module, Optimizer의 역할을 이해한다
- PyTorch 학습 루프의 5단계 정형 패턴을 몸에 익힌다


In [ ]:
import torch
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"

# ── 1. Tensor & Autograd ──
x = torch.tensor([2.0], requires_grad=True)
y = x ** 2 + 3 * x          # y = x² + 3x
y.backward()
print(x.grad)               # dy/dx = 2x + 3 = 7

# ── 2. nn.Module로 Q-네트워크 정의 ──
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, action_dim),
        )
    def forward(self, s):
        return self.net(s)      # 각 행동의 Q값 벡터

# ── 3. 학습 루프 5단계 (회귀 예제로 패턴 익히기) ──
model = QNetwork(4, 2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

states = torch.randn(64, 4, device=device)       # 가짜 배치
targets = torch.randn(64, 2, device=device)

for step in range(200):
    pred = model(states)             # ① 순전파
    loss = criterion(pred, targets)  # ② 손실
    optimizer.zero_grad()            # ③ 기울기 초기화
    loss.backward()                  # ④ 역전파
    optimizer.step()                 # ⑤ 갱신
    if step % 50 == 0:
        print(f"step {step:3d}  loss = {loss.item():.4f}")

## 4교시 · DQN, Double DQN 구현

`13:30 ~ 14:30` · `dqn_cartpole.py`

> ⏳ 실행에 약 52초 걸립니다

- CartPole-v1에서 DQN 전체 파이프라인을 완성한다
- 플래그 하나로 Double DQN으로 전환해 성능을 비교한다


In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import numpy as np

env = gym.make("CartPole-v1")
obs_dim = env.observation_space.shape[0]     # 4
n_actions = env.action_space.n               # 2

q_net = QNetwork(obs_dim, n_actions)
q_target = QNetwork(obs_dim, n_actions)
q_target.load_state_dict(q_net.state_dict())
optimizer = torch.optim.Adam(q_net.parameters(), lr=1e-3)
buffer = ReplayBuffer(50_000)

gamma, batch_size = 0.99, 64
eps, eps_min, eps_decay = 1.0, 0.05, 0.995
DOUBLE = True                                # ← Double DQN 스위치

def train_step():
    s, a, r, s_next, done = buffer.sample(batch_size)
    q = q_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
    with torch.no_grad():
        if DOUBLE:   # 선택은 온라인넷, 평가는 타깃넷
            best_a = q_net(s_next).argmax(1, keepdim=True)
            q_next = q_target(s_next).gather(1, best_a).squeeze(1)
        else:        # 타깃넷이 선택+평가 (바닐라 DQN)
            q_next = q_target(s_next).max(1).values
        target = r + gamma * q_next * (1 - done)
    loss = nn.functional.smooth_l1_loss(q, target)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

returns = []
for episode in range(400):
    s, _ = env.reset()
    total, done = 0, False
    while not done:
        if np.random.rand() < eps:
            a = env.action_space.sample()
        else:
            with torch.no_grad():
                a = q_net(torch.as_tensor(s, dtype=torch.float32)).argmax().item()
        s_next, r, term, trunc, _ = env.step(a)
        done = term or trunc
        buffer.push(s, a, r, s_next, float(term))
        s, total = s_next, total + r
        if len(buffer) >= 1000:
            train_step()
    eps = max(eps_min, eps * eps_decay)
    if episode % 20 == 0:                    # 타깃 네트워크 동기화
        q_target.load_state_dict(q_net.state_dict())
    returns.append(total)
    if episode % 20 == 0:
        print(f"ep {episode:3d}  return {np.mean(returns[-20:]):6.1f}  eps {eps:.2f}")
# 평균 리턴이 475를 넘으면 CartPole 해결!

## 5교시 · Policy Gradient 소개

`14:30 ~ 15:30` · `reinforce_cartpole.py`

> ⏳ 실행에 약 1분 17초 걸립니다 (학습 루프 — 멈춘 것이 아닙니다)

- 가치 기반과 정책 기반 접근의 차이를 이해한다
- REINFORCE 알고리즘과 로그-미분 트릭을 이해한다


In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import numpy as np

env = gym.make("CartPole-v1")

policy = nn.Sequential(
    nn.Linear(4, 128), nn.ReLU(),
    nn.Linear(128, 2),          # 행동별 로짓
)
optimizer = torch.optim.Adam(policy.parameters(), lr=1e-3)
gamma = 0.99

for episode in range(600):
    s, _ = env.reset()
    log_probs, rewards, done = [], [], False
    while not done:                              # 1) 에피소드 수집
        logits = policy(torch.as_tensor(s, dtype=torch.float32))
        dist = torch.distributions.Categorical(logits=logits)
        a = dist.sample()
        log_probs.append(dist.log_prob(a))
        s, r, term, trunc, _ = env.step(a.item())
        done = term or trunc
        rewards.append(r)

    G, returns = 0.0, []                         # 2) 리턴 계산 (뒤에서부터)
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    returns = torch.tensor(returns)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)  # 정규화(간이 베이스라인)

    loss = -(torch.stack(log_probs) * returns).sum()   # 3) ∇logπ · G
    optimizer.zero_grad(); loss.backward(); optimizer.step()

    if episode % 50 == 0:
        print(f"ep {episode:3d}  return {sum(rewards):.0f}")

## 6교시 · Actor-Critic 소개

`15:30 ~ 16:30` · `actor_critic_net.py`

- Actor(정책)와 Critic(가치)의 역할 분담을 이해한다
- 어드밴티지 함수와 A2C의 구조를 설명할 수 있다


In [ ]:
import torch
import torch.nn as nn

class ActorCritic(nn.Module):
    """몸통을 공유하고 머리만 둘 — A2C의 표준 구조"""
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
        )
        self.actor_head = nn.Linear(hidden, action_dim)   # 정책 로짓
        self.critic_head = nn.Linear(hidden, 1)           # V(s)

    def forward(self, s):
        h = self.body(s)
        return self.actor_head(h), self.critic_head(h).squeeze(-1)

# 손실 구성 미리보기 (다음 교시에 전체 루프 완성)
def a2c_loss(logits, value, action, td_target, entropy_coef=0.01):
    dist = torch.distributions.Categorical(logits=logits)
    advantage = (td_target - value).detach()      # Critic 신호는 Actor로 역전파 금지
    actor_loss = -(dist.log_prob(action) * advantage).mean()
    critic_loss = nn.functional.mse_loss(value, td_target)
    entropy = dist.entropy().mean()               # 탐험 유지 보너스
    return actor_loss + 0.5 * critic_loss - entropy_coef * entropy

## 7교시 · A2C 구현

`16:30 ~ 17:30` · `a2c_cartpole.py`

> ⏳ 실행에 약 16초 걸립니다

- CartPole-v1에서 n-step A2C를 완성한다
- DQN·REINFORCE와 학습 속도·안정성을 비교한다


In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import numpy as np

env = gym.make("CartPole-v1")
model = ActorCritic(4, 2)
optimizer = torch.optim.Adam(model.parameters(), lr=7e-4)
gamma, n_steps = 0.99, 5

s, _ = env.reset()
ep_return, returns = 0, []
for update in range(3000):
    # ── n-step 롤아웃 수집 ──
    log_probs, values, rewards, entropies, dones = [], [], [], [], []
    for _ in range(n_steps):
        logits, v = model(torch.as_tensor(s, dtype=torch.float32))
        dist = torch.distributions.Categorical(logits=logits)
        a = dist.sample()
        s_next, r, term, trunc, _ = env.step(a.item())
        done = term or trunc
        log_probs.append(dist.log_prob(a)); values.append(v)
        rewards.append(r); dones.append(done)
        entropies.append(dist.entropy())
        ep_return += r
        s = s_next
        if done:
            returns.append(ep_return); ep_return = 0
            s, _ = env.reset()

    # ── n-step 리턴으로 TD 목표 계산 ──
    with torch.no_grad():
        _, v_last = model(torch.as_tensor(s, dtype=torch.float32))
    R, td_targets = v_last, []
    for r, d in zip(reversed(rewards), reversed(dones)):
        R = r + gamma * R * (1 - d)
        td_targets.insert(0, R)
    td_targets = torch.stack(td_targets).detach()
    values = torch.stack(values)
    advantages = td_targets - values

    actor_loss = -(torch.stack(log_probs) * advantages.detach()).mean()
    critic_loss = advantages.pow(2).mean()
    entropy = torch.stack(entropies).mean()
    loss = actor_loss + 0.5 * critic_loss - 0.01 * entropy

    optimizer.zero_grad(); loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 0.5)   # 기울기 폭주 방지
    optimizer.step()

    if update % 200 == 0 and returns:
        print(f"update {update:4d}  최근 20ep 평균 {np.mean(returns[-20:]):6.1f}")